# 7.4 Loss Functions and Optimization

[Back to neural networks index](../7_neural_networks.ipynb)

This notebook focuses on the objective functions and optimization procedures that make learning happen. Neural networks do not update themselves; they move by following the loss surface.


## 7.4.1 MSE, binary cross-entropy, and multiclass cross-entropy

### Why It Matters
Different tasks need different losses. The fastest way to see the distinction is to compute them on small toy predictions.


In [ ]:
import torch
from torch import nn

mse = nn.MSELoss()(torch.tensor([[2.2], [1.8]]), torch.tensor([[2.0], [2.0]])).item()
bce = nn.BCELoss()(torch.tensor([[0.8], [0.3]]), torch.tensor([[1.0], [0.0]])).item()
ce = nn.CrossEntropyLoss()(torch.tensor([[2.0, 0.2, -1.0], [0.1, 1.3, -0.2]]), torch.tensor([0, 1])).item()
{"mse": round(mse, 4), "binary_cross_entropy": round(bce, 4), "cross_entropy": round(ce, 4)}


### Interpretation
The loss has to match the output semantics of the model. A wrong loss can make a correct-looking architecture learn the wrong thing.


## 7.4.2 Gradients and gradient descent intuition

### Why It Matters
A gradient tells us which local direction increases loss. Gradient descent steps in the opposite direction. On a one-parameter problem, the movement is easy to inspect directly.


In [ ]:
import torch

w = torch.tensor([5.0], requires_grad=True)
lr = 0.2
path = []
for _ in range(5):
    loss = (w - 1.5).pow(2).sum()
    loss.backward()
    with torch.no_grad():
        path.append((round(float(w.item()), 3), round(float(loss.item()), 3)))
        w -= lr * w.grad
    w.grad.zero_()
path


### Interpretation
The parameter walks downhill toward the minimizer. Real networks do the same thing in much higher-dimensional parameter spaces.


## 7.4.3 SGD versus Adam

### Why It Matters
Optimizers differ in how they use gradient history. On the same toy classification problem, they can reach similar accuracy with different learning dynamics.


In [ ]:
import copy
import torch
from torch import nn
from sklearn.datasets import make_moons

X_np, y_np = make_moons(n_samples=200, noise=0.18, random_state=15)
X = torch.tensor(X_np, dtype=torch.float32)
y = torch.tensor(y_np.reshape(-1, 1), dtype=torch.float32)

base = nn.Sequential(nn.Linear(2, 12), nn.Tanh(), nn.Linear(12, 1), nn.Sigmoid())
loss_fn = nn.BCELoss()
results = {}
for name, optimizer_cls, lr in [("SGD", torch.optim.SGD, 0.1), ("Adam", torch.optim.Adam, 0.03)]:
    model = copy.deepcopy(base)
    opt = optimizer_cls(model.parameters(), lr=lr)
    for _ in range(150):
        opt.zero_grad()
        loss = loss_fn(model(X), y)
        loss.backward()
        opt.step()
    with torch.no_grad():
        acc = (((model(X) > 0.5).float() == y).float().mean()).item()
    results[name] = {"loss": round(float(loss.item()), 4), "accuracy": round(float(acc), 3)}
results


### Interpretation
Adam often converges faster with less tuning, while SGD remains conceptually simpler and sometimes preferable for controlled experiments.


## 7.4.4 Learning-rate effects

### Why It Matters
A learning rate that is too small moves painfully slowly. A learning rate that is too large can overshoot or bounce. This cell compares one-step trajectories.


In [ ]:
import torch

def run(lr):
    w = torch.tensor([4.0], requires_grad=True)
    losses = []
    for _ in range(6):
        loss = (w - 0.5).pow(2).sum()
        loss.backward()
        with torch.no_grad():
            losses.append(round(float(loss.item()), 3))
            w -= lr * w.grad
        w.grad.zero_()
    return losses

{"lr_0.05": run(0.05), "lr_0.4": run(0.4), "lr_1.1": run(1.1)}


### Interpretation
The learning rate is one of the most important hyperparameters because it directly controls stability and speed.


## 7.4.5 Train-step anatomy

### Why It Matters
A single training step contains the whole learning loop in miniature: forward pass, loss, backward pass, and optimizer step.


In [ ]:
import torch
from torch import nn

torch.manual_seed(16)
X = torch.randn(10, 3)
y = torch.randn(10, 1)
model = nn.Linear(3, 1)
opt = torch.optim.SGD(model.parameters(), lr=0.05)
loss_fn = nn.MSELoss()

opt.zero_grad()
preds = model(X)
loss = loss_fn(preds, y)
loss.backward()
grad_norm = round(float(model.weight.grad.norm().item()), 4)
opt.step()
{"loss": round(float(loss.item()), 4), "weight_grad_norm": grad_norm}


### Interpretation
If you understand one train step clearly, you understand the backbone of every larger training loop.


## 7.4.6 Compare optimizer behavior on a toy problem

### Why It Matters
The easiest way to compare optimizers is to give them the same model and data and log how their losses move over early updates.


In [ ]:
import copy
import torch
from torch import nn

torch.manual_seed(17)
X = torch.randn(120, 4)
y = (X[:, :2].sum(dim=1, keepdim=True) > 0).float()
base = nn.Sequential(nn.Linear(4, 10), nn.ReLU(), nn.Linear(10, 1), nn.Sigmoid())
loss_fn = nn.BCELoss()

trajectories = {}
for name, opt_cls, lr in [("SGD", torch.optim.SGD, 0.05), ("Adam", torch.optim.Adam, 0.02)]:
    model = copy.deepcopy(base)
    opt = opt_cls(model.parameters(), lr=lr)
    hist = []
    for _ in range(20):
        opt.zero_grad()
        loss = loss_fn(model(X), y)
        loss.backward()
        opt.step()
        hist.append(round(float(loss.item()), 4))
    trajectories[name] = hist[:6]
trajectories


### Interpretation
Comparing the early trajectory is often more informative than looking only at the last epoch.
